# Mission 3 — 실험, 학습 코드, 실행 기록

RoBERTa-base의 선택적 양성 BCE 가중치(오심2.0·두통1.5)와 Basic512 입력을 사용한 후보다. 모델 입력은 전사뿐이며 모든 threshold는0.5다. 이 노트북은 학습을 자동 시작하지 않는다.

## 학습 코드
아래 셀은 train.py의 함수 정의다. 실제 재학습에는 검증된 주최 측 데이터와 공개 원본 pretrained 모델 경로를 명시한다. 이전62,362건 앙상블을 다시 학습하거나 포함하지 않는다.

In [1]:
"""Portable reference for the reported focused-positive-weight BCE experiments.
Fresh training requires an independently verified provided-data list and local public pretrained weights.
This module does not launch training during inference or notebook record review.
"""
import argparse,csv,json,math,os,random,time
from pathlib import Path
os.environ['TOKENIZERS_PARALLELISM']='false'
import numpy as np
import torch
from torch.utils.data import Dataset,DataLoader
from transformers import AutoTokenizer,AutoModelForSequenceClassification,DataCollatorWithPadding,get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from model import LABELS,POSITIVE_WEIGHTS,MAX_LENGTH,THRESHOLD,text_only,probabilities

class Rows(Dataset):
    def __init__(self,texts,y,tok):self.texts=texts;self.y=y;self.tok=tok
    def __len__(self):return len(self.texts)
    def __getitem__(self,i):return dict(self.tok(self.texts[i],truncation=True,max_length=MAX_LENGTH),labels=self.y[i].astype('float32').tolist())

def read_data(path):
    texts,labels=[],[];path=Path(path)
    if path.is_file() and path.suffix=='.csv':
        with path.open(encoding='utf-8-sig',newline='') as f:
            for row in csv.DictReader(f):
                texts.append(str(row.get('text') or ''));labels.append([int(float(row[l])) for l in LABELS])
    else:
        for file in sorted(path.rglob('*.json')):
            raw=json.loads(file.read_text(encoding='utf-8-sig'));texts.append(text_only(raw))
            targets=raw.get('symptom')
            if targets is None:targets=raw.get('metadata',{}).get('symptom',[])
            if isinstance(targets,str):targets=[targets]
            labels.append([int(l in (targets or [])) for l in LABELS])
    if not texts:raise ValueError('No data')
    # No sample is removed based on symptoms. Only labels outside LABELS are ignored.
    return texts,np.asarray(labels,dtype=np.float32)

@torch.no_grad()
def evaluate(model,dl,device):
    model.eval();ps=[];ys=[]
    for batch in dl:
        ys.append(batch.pop('labels').numpy());ps.append(probabilities(model,batch,device).numpy())
    p=np.concatenate(ps);y=np.concatenate(ys)
    return float(f1_score(y,p>=THRESHOLD,average='macro',zero_division=0)),p

def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--train_data',type=Path,required=True,help='Verified supplied Seoul training JSON folder or CSV')
    ap.add_argument('--validation_data',type=Path,required=True,help='Verified supplied Validation JSON folder or CSV; evaluation only')
    ap.add_argument('--pretrained_dir',type=Path,required=True,help='Fresh public klue/roberta-base stored locally')
    ap.add_argument('--output_dir',type=Path,required=True)
    ap.add_argument('--seed',type=int,required=True,help='Set explicitly; source notebook code/output disagree on seed')
    args=ap.parse_args()
    if args.train_data.resolve()==args.validation_data.resolve():raise ValueError('Training and Validation must be separate')
    texts,y=read_data(args.train_data);vt,vy=read_data(args.validation_data)
    if (len(texts),len(vt))!=(29200,3640):raise ValueError('Reported experiment expects29200/3640 supplied files; do not filter labels to force these counts')
    random.seed(args.seed);np.random.seed(args.seed);torch.manual_seed(args.seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(args.seed)
    torch.set_num_threads(6);device='cuda' if torch.cuda.is_available() else 'cpu'
    tok=AutoTokenizer.from_pretrained(args.pretrained_dir,local_files_only=True)
    model=AutoModelForSequenceClassification.from_pretrained(args.pretrained_dir,num_labels=9,problem_type='multi_label_classification',local_files_only=True).to(device)
    collate=DataCollatorWithPadding(tok,pad_to_multiple_of=8)
    td=DataLoader(Rows(texts,y,tok),batch_size=8,shuffle=True,collate_fn=collate)
    vd=DataLoader(Rows(vt,vy,tok),batch_size=16,shuffle=False,collate_fn=collate)
    opt=torch.optim.AdamW(model.parameters(),lr=2e-5,weight_decay=.01)
    steps=math.ceil(len(td)/2);schedule=get_linear_schedule_with_warmup(opt,int(steps*8*.1),steps*8)
    scaler=torch.amp.GradScaler('cuda',enabled=device=='cuda')
    loss_fn=torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POSITIVE_WEIGHTS,device=device))
    args.output_dir.mkdir(parents=True,exist_ok=True)
    history=[];best=-1.;bad=0
    for epoch in range(1,9):
        model.train();opt.zero_grad();losses=0.;started=time.perf_counter()
        for i,batch in enumerate(td):
            target=batch.pop('labels').to(device)
            with torch.autocast(device_type=device,dtype=torch.float16,enabled=device=='cuda'):
                logits=model(**{k:v.to(device) for k,v in batch.items()}).logits
                raw_loss=loss_fn(logits,target);loss=raw_loss/2
            if not torch.isfinite(loss):raise RuntimeError('Nonfinite loss')
            scaler.scale(loss).backward();losses+=raw_loss.item()
            if (i+1)%2==0 or i+1==len(td):
                scaler.step(opt);scaler.update();schedule.step();opt.zero_grad()
        score,p=evaluate(model,vd,device)
        row={'epoch':epoch,'train_loss':losses/len(td),'macro_f1':score,'seconds':time.perf_counter()-started,'threshold':THRESHOLD,'seed':args.seed}
        history.append(row);(args.output_dir/'history.json').write_text(json.dumps(history,indent=2));print(row,flush=True)
        if score>best:
            best=score;bad=0;dest=args.output_dir/'model';dest.mkdir(exist_ok=True)
            model.config.save_pretrained(dest);tok.save_pretrained(dest);torch.save(model.state_dict(),dest/'model.pt')
            np.save(args.output_dir/'validation_probabilities.npy',p);(args.output_dir/'best.json').write_text(json.dumps(row,indent=2))
        else:bad+=1
        if bad>=2:break



/projects/DCC_2026/.conda/mission3-next/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 원본 학습 실행 출력
제공된 노트북의 실제 저장 출력을 그대로 읽는다. 여기 남은 run의 최고는0.623516이며, 문서 최고0.638604의 별도 run과 완전히 대응한다고 주장하지 않는다. 코드의 seed123과 출력2026 표기 차이도 원본 기록에 남아 있다.

In [2]:
from pathlib import Path
import json
records = Path('records')
print((records / 'original_training_log.txt').read_text())


=== Original notebook cell 12; preserved actual stored output ===
Seed           : 2026
Model          : klue/roberta-base
Labels         : 9
Max length     : 512
Train batch    : 8
Eval batch     : 16
Effective batch: 16
Learning rate  : 2e-05
Epochs         : 8
Threshold      : 0.5

=== Original notebook cell 14; preserved actual stored output ===
Existing Training CSV found.
전처리를 수행하지 않고 기존 CSV를 사용합니다.

Training Dataset
Samples      : 29,200
Label count  : 9
Null text    : 0
All-negative : 0

Class distribution
고열      5,186 (17.76%)
구토      4,463 (15.28%)
두통      2,905 ( 9.95%)
복통      6,771 (23.19%)
어지러움    6,334 (21.69%)
열상      3,465 (11.87%)
오심      3,341 (11.44%)
전신쇠약    5,310 (18.18%)
호흡곤란    3,927 (13.45%)

=== Original notebook cell 20; preserved actual stored output ===
Existing Validation CSV found.
전처리를 수행하지 않고 기존 CSV를 사용합니다.

Validation Dataset
Samples      : 3,640
Label shape  : (3640, 9)
All-negative : 0

=== Original notebook cell 24; preserved actual stored output 

## 오류 분석과 설정 선택
문서 기준 BCE0.611252→오심 가중치0.627050→두통 추가0.635403. 구토 추가0.631112, Head+Tail0.633951은 제외했다. 세 반복은0.635403/0.638604/0.623516이다. 다른 범위의 재추출 실험 수치와 직접 비교하지 않는다. 상세 실험 문서는 records/EXPERIMENTS.md에 포함했다.

In [3]:
print(json.dumps(json.loads((records / 'model_provenance.json').read_text()), ensure_ascii=False, indent=2))

{
  "format": "mission3-roberta-base-weighted-bce-v1",
  "labels": [
    "고열",
    "구토",
    "두통",
    "복통",
    "어지러움",
    "열상",
    "오심",
    "전신쇠약",
    "호흡곤란"
  ],
  "threshold": 0.5,
  "positive_loss_weights": [
    1.0,
    1.0,
    1.5,
    1.0,
    1.0,
    1.0,
    2.0,
    1.0,
    1.0
  ],
  "parameters": 110625033,
  "source_safetensors_sha256": "abda46dcd11fd62c406e57330fb01ae6ba3343077a8b5975e09f91a88ec61704",
  "submitted_pt_sha256": "f6621239dd698a0ba1d0fd9aac7af84a0f3d098462ec11dc0f51ef88abab500f",
  "document_best_f1": 0.638604,
  "positive_cached_recheck_f1": 0.6385532655493376,
  "notebook_stored_run_f1": 0.623516,
  "document_training_count": 29200,
  "document_validation_count": 3640,
  "official_dataset_membership_verified": false,
  "best_training_run_log_correspondence_verified": false,
  "source_archive": "submission_archive/mission3_before_final_20261012T084824Z",
  "tensor_conversion": "All source tensors copied without modification; not retrained."
}


## 제출 패키지 재확인
인터넷 차단 GPU·CPU 공식 CLI와 metadata 불변성을 확인했다. 기존 양성3,640건 토큰 캐시 점수0.638553은 문서 최고에 가깝다. 공식 배포 파일 목록 검증과 학습 run의 로그 대응은 아직 미확인이다.

In [4]:
print(json.dumps(json.loads((records / 'final_validation_verification.json').read_text()), ensure_ascii=False, indent=2))
print(json.dumps(json.loads((records / 'final_cli_verification.json').read_text()), ensure_ascii=False, indent=2))

{
  "samples": 7748,
  "positive_only_samples": 3640,
  "scope": "Existing cached local validation; official supplied membership remains unverified",
  "positive_only_macro_f1": 0.6385532655493376,
  "full_cached_macro_f1": 0.4554556494076007,
  "cached_forward_seconds_excluding_load": 15.438015569990966,
  "threshold": 0.5,
  "batch_size": 16,
  "gpu": "NVIDIA GeForce RTX 5070",
  "total_parameters": 110625033,
  "active_parameters": 110625033,
  "source_probability_path_exact_on_checked_inputs": true
}
{
  "checks": [
    {
      "device": "cuda",
      "variant": 0,
      "rows": 4,
      "stdout": "{\"samples\": 4, \"batch_size\": 16, \"device\": \"cuda\", \"gpu\": \"NVIDIA GeForce RTX 5070\", \"python\": \"3.11.16\", \"torch\": \"2.8.0+cu128\", \"total_parameters\": 110625033, \"active_parameters\": 110625033, \"threshold\": 0.5, \"forward_with_tokenization_seconds\": 0.1286616370198317, \"total_seconds\": 0.876634264976019, \"seconds_per_sample\": 0.21915856624400476, \"precision